In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('Titanic Dataset.csv')

df_base = df.copy()
cols_to_drop = ['cabin', 'boat', 'body', 'home.dest', 'ticket', 'name']
df_base = df_base.drop(columns=cols_to_drop)

df_base['age'] = df_base['age'].fillna(df_base['age'].mean())
df_base['fare'] = df_base['fare'].fillna(df_base['fare'].mean())
df_base['embarked'] = df_base['embarked'].fillna(df_base['embarked'].mode()[0])

X_base = pd.get_dummies(df_base.drop('survived', axis=1), drop_first=True)
y_base = df_base['survived']

df_adv = df.copy()
df_adv['cabin_missing'] = df_adv['cabin'].isnull().astype(int)
df_adv['title'] = df_adv['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df_adv['age'] = df_adv.groupby('title')['age'].transform(lambda x: x.fillna(x.median()))
df_adv['age'] = df_adv['age'].fillna(df_adv['age'].median())

df_adv = df_adv.drop(columns=['cabin', 'boat', 'body', 'home.dest', 'ticket', 'name', 'title'])
df_adv['fare'] = df_adv['fare'].fillna(df_adv['fare'].mean())
df_adv['embarked'] = df_adv['embarked'].fillna(df_adv['embarked'].mode()[0])

X_adv = pd.get_dummies(df_adv.drop('survived', axis=1), drop_first=True)
y_adv = df_adv['survived']

def evaluate_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)
    return accuracy_score(y_test, model.predict(X_test_scaled))

acc_base = evaluate_model(X_base, y_base)
acc_adv = evaluate_model(X_adv, y_adv)

print(f"Dokładność (Accuracy) Baseline: {acc_base:.4f}")
print(f"Dokładność (Accuracy) Zaawansowane: {acc_adv:.4f}")